# Calculate scTab embeddings

In [4]:
!pip install -e /dss/dsshome1/04/di93zer/git/cellnet --no-deps

Obtaining file:///dss/dsshome1/04/di93zer/git/cellnet
  Preparing metadata (setup.py) ... done
  Running setup.py develop for cellnet


In [1]:
import os
from os.path import join

import anndata
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import tqdm

/usr/local/lib/python3.8/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
VAR_FILE = "/dss/dssmcmlfs01/pn36po/pn36po-dss-0000/di93zer/merlin_cxg_2023_05_15_sf-log1p/var.parquet"
CKPT = "/mnt/dssfs02/tb_logs/cxg_2023_05_15_tabnet/default/w_augment_4/checkpoints/val_f1_macro_epoch=45_val_f1_macro=0.847.ckpt"
HPARAMS = "/mnt/dssfs02/tb_logs/cxg_2023_05_15_tabnet/default/w_augment_4/hparams.yaml"

In [3]:
from scipy.sparse import csc_matrix
from cellnet.utils.data_loading import streamline_count_matrix


def get_aligned_feature_space(adata, genes_model):
    adata = adata[:, adata.var.feature_name.isin(genes_from_model.feature_name).to_numpy()]
    x_streamlined = streamline_count_matrix(
        csc_matrix(adata.X), 
        adata.var.feature_name,
        genes_from_model.feature_name
    )
    return x_streamlined


genes_from_model = pd.read_parquet(VAR_FILE)

In [4]:
def sf_log1p_norm(x):
    """Normalize each cell to have 10000 counts and apply log(x+1) transform."""

    counts = torch.sum(x, dim=1, keepdim=True)
    # avoid zero division error
    counts += counts == 0.
    scaling_factor = 10000. / counts

    return torch.log1p(scaling_factor * x)

In [5]:
from collections import OrderedDict
import yaml

from cellnet.tabnet.tab_network import TabNet


ckpt = torch.load(CKPT, map_location=torch.device('cuda'))
tabnet_weights = OrderedDict()
for name, weight in ckpt['state_dict'].items():
    if 'classifier.' in name:
        tabnet_weights[name.replace('classifier.', '')] = weight
# load in hparams file of model to get model architecture
with open(HPARAMS) as f:
    model_params = yaml.full_load(f.read())
# initialzie model with hparams from hparams.yaml file
tabnet = TabNet(
    input_dim=model_params['gene_dim'],
    output_dim=model_params['type_dim'],
    n_d=model_params['n_d'],
    n_a=model_params['n_a'],
    n_steps=model_params['n_steps'],
    gamma=model_params['gamma'],
    n_independent=model_params['n_independent'],
    n_shared=model_params['n_shared'],
    epsilon=model_params['epsilon'],
    virtual_batch_size=model_params['virtual_batch_size'],
    momentum=model_params['momentum'],
    mask_type=model_params['mask_type'],
)
# load trained weights
tabnet.load_state_dict(tabnet_weights)
tabnet.to("cuda")
# set model to inference mode
tabnet.eval();

In [6]:
DATA_PATH = "/mnt/dssfs02/dataset-similarity/preprocessed"

In [7]:
from cellnet.utils.data_loading import dataloader_factory


for file in tqdm.tqdm(sorted(os.listdir(DATA_PATH))):
    adata = sc.read_h5ad(join(DATA_PATH, file))
    
    n_cells = len(adata)
    x_embed = []
    for idxs in np.array_split(np.arange(n_cells), max(1, n_cells / 200_000)):
        loader = dataloader_factory(
            get_aligned_feature_space(adata[idxs, :], genes_from_model), 
            batch_size=2048
        )
        with torch.no_grad():
            for batch in loader:
                x_input = sf_log1p_norm(batch[0]["X"].cuda())
                steps_output, _ = tabnet.encoder(x_input)
                x_embed.append(
                    torch.sum(torch.stack(steps_output, dim=0), dim=0)
                    .detach()
                    .cpu()
                    .numpy()
                )
    adata.obsm["X_scTab"] = np.vstack(x_embed)
    adata.write_h5ad(join(DATA_PATH, file))


100%|██████████| 18/18 [1:03:28<00:00, 211.61s/it]
